# MergeKit GA Notebook
This notebook helps you launch and monitor MergeKit GA runs without leaving Jupyter. Configure the run parameters, execute the CLI from a code cell, and inspect the generated artifacts (history CSV, best config, merged outputs) directly from the notebook.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the mergekit repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / "workspace" / "hf-cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(PROJECT_ROOT / "workspace" / "transformers_cache"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Storage base: {PROJECT_ROOT / 'workspace'}")

## Environment Check
Ensure you are using the same virtual environment that you use for local runs. If packages are missing, run `%pip install -e .[evolve-ga]` in the next cell (uncomment first).

In [ ]:
# Uncomment the next line if you need to (re)install mergekit extras in this environment.
# %pip install -e .[evolve-ga]

## Run Configuration
Set the GA configuration file, storage location, and runtime options you want to experiment with. These defaults reproduce the CPU-only run for `examples/experiment1.yml`.

In [ ]:
from datetime import datetime

CONFIG_PATH = PROJECT_ROOT / "examples" / "experiment1.yml"
STORAGE_ROOT = PROJECT_ROOT / "workspace" / "ga-notebook"
RUN_LABEL = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_PATH = STORAGE_ROOT / RUN_LABEL

# GA parameters (tweak as needed)
MAX_FEVALS = 120
STRATEGY = "serial"  # options: serial | pool | buffered
NUM_GPUS = 0
NUM_WORKERS = 4  # used when STRATEGY != serial or when running CPU pool/buffered
MERGE_CUDA = False
ALLOW_BENCHMARK_TASKS = False  # keep False unless you really know what you're doing
TRUST_REMOTE_CODE = False
BATCH_SIZE = None  # set to an int to override lm-eval batch size
RANDOM_SEED = 0
TIMEOUT = None  # seconds; set to a float to limit wall-clock time
EXTRA_ARGS = []  # supply additional CLI switches as strings here

print(f"Config file: {CONFIG_PATH}")
print(f"Run output will be written to: {RUN_PATH}")

In [ ]:
import shlex
import subprocess
from typing import Iterable, Optional

def run_ga(
    config_path: Path,
    storage_path: Path,
    *,
    max_fevals: int = 100,
    strategy: str = "serial",
    num_gpus: Optional[int] = None,
    num_workers: Optional[int] = None,
    merge_cuda: bool = True,
    allow_benchmark_tasks: bool = False,
    trust_remote_code: bool = False,
    batch_size: Optional[int] = None,
    random_seed: Optional[int] = None,
    timeout: Optional[float] = None,
    extra_args: Optional[Iterable[str]] = None,
 ) -> Path:
    """Invoke mergekit-evolve-ga via the Python interpreter and stream output."""
    storage_path = storage_path.resolve()
    storage_path.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "-m","mergekit.scripts.evolve_ga",
        str(config_path),
        "--storage-path",
        str(storage_path),
        "--strategy",
        strategy,
        "--max-fevals",
        str(max_fevals),
    ]
    if num_gpus is not None:
        cmd.extend(["--num-gpus", str(num_gpus)])
    if num_workers is not None and strategy in {"pool", "buffered"}:
        cmd.extend(["--num-workers", str(num_workers)])
    if merge_cuda:
        cmd.append("--merge-cuda")
    else:
        cmd.append("--no-merge-cuda")
    if allow_benchmark_tasks:
        cmd.append("--i-understand-the-depths-of-the-evils-i-am-unleashing")
    if trust_remote_code:
        cmd.append("--trust-remote-code")
    if batch_size is not None:
        cmd.extend(["--batch-size", str(batch_size)])
    if random_seed is not None:
        cmd.extend(["--random-seed", str(random_seed)])
    if timeout is not None:
        cmd.extend(["--timeout", str(timeout)])
    if extra_args:
        cmd.extend(list(extra_args))

    print("Running command:")
    print(" ".join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"mergekit-evolve-ga exited with code {result.returncode}")
    return storage_path

In [ ]:
RUN_PATH = RUN_PATH.resolve()
RUN_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Launching GA run in {RUN_PATH}")

run_output_path = run_ga(
    CONFIG_PATH,
    RUN_PATH,
    max_fevals=MAX_FEVALS,
    strategy=STRATEGY,
    num_gpus=NUM_GPUS,
    num_workers=NUM_WORKERS,
    merge_cuda=MERGE_CUDA,
    allow_benchmark_tasks=ALLOW_BENCHMARK_TASKS,
    trust_remote_code=TRUST_REMOTE_CODE,
    batch_size=BATCH_SIZE,
    random_seed=RANDOM_SEED,
    timeout=TIMEOUT,
    extra_args=EXTRA_ARGS,
 )
print("GA run complete.")
print(f"Artifacts stored under: {run_output_path}")

## Inspect Results
After the run finishes, use the helpers below to explore the GA history and best-found configuration.

In [ ]:
import pandas as pd

history_file = run_output_path / "ga_history.csv"
if history_file.exists():
    df_hist = pd.read_csv(history_file)
    display(df_hist.tail())
else:
    print(f"No ga_history.csv found in {run_output_path}")

In [ ]:
best_config_file = run_output_path / "best_config.yaml"
if best_config_file.exists():
    print(best_config_file.read_text())
else:
    print(f"No best_config.yaml found in {run_output_path}")

In [ ]:
from itertools import islice

def list_directory(path: Path, depth: int = 1, max_entries: int = 50):
    path = Path(path)
    if not path.exists():
        print(f"{path} does not exist")
        return
    print(f"Listing {path} (depth={depth})")
    for entry in islice(sorted(path.iterdir()), 0, max_entries):
        print(entry.relative_to(path.parent))
        if depth > 1 and entry.is_dir():
            for child in islice(sorted(entry.iterdir()), 0, max_entries):
                print(f"  {child.relative_to(path.parent)}")

list_directory(run_output_path, depth=2)